In [ ]:
#Start by connecting gdrive into the google colab

from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/MyDrive
!ls
#!unzip "/content/drive/MyDrive/ourdataset.zip" -d "/content/drive/MyDrive/MachineLearning/"

In [ ]:
!pip install pycm==0.8.1

In [ ]:
%cd /content/drive/MyDrive/MachineLearning/ourdataset/train/normal/
!ls

In [ ]:
!ls /content/drive/MyDrive/MachineLearning/ourdataset/train/

abnormal  mel_features_svm_a.data  mel_features_svm_b.data  normal


In [ ]:
#Split 3 folds
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
# importing shutil module  
import shutil

def copyTrain(data, k):
    for i in range(len(data)):
        source = data[i]
        destination = data[i].replace("train/", "train/train_kfold"+str(k))
        #dest = shutil.move(source, destination)
        dest = shutil.copy(source, destination) 

def copyTest(data, k):
    for i in range(len(data)):
        source = data[i]
        destination = data[i].replace("train/", "train/test_kfold"+str(k))
        #dest = shutil.move(source, destination)
        dest = shutil.copy(source, destination) 
        
cross_number = 3
kf = KFold(n_splits = cross_number, shuffle = True, random_state=10)
folder = "/content/drive/MyDrive/MachineLearning/ourdataset/train/"
normal = np.array(glob(folder+'/normal/*.wav'))
abnormal = np.array(glob(folder+'/abnormal/*.wav'))

#normal
kf.get_n_splits(normal)
i=1
for normal_train_index, normal_test_index in kf.split(normal):
    normal_train = normal[normal_train_index]
    normal_test = normal[normal_test_index]


    if not os.path.exists(folder+"train_kfold"+str(i)) and not os.path.exists(folder+"test_kfold"+str(i)):
        os.mkdir(folder+"train_kfold"+str(i))
        os.mkdir(folder+"test_kfold"+str(i))

    if (not os.path.exists(folder+"train_kfold"+str(i)+"/normal") and not os.path.exists(folder+"test_kfold"+str(i)+"/normal")):
        os.mkdir(folder+"train_kfold"+str(i)+"/normal")
        os.mkdir(folder+"test_kfold"+str(i)+"/normal")

    copyTrain(normal_train, i)
    copyTest(normal_test, i)
    
    i+=1
    
#abnormal
kf.get_n_splits(abnormal)
i=1
for abnormal_train_index, abnormal_test_index in kf.split(abnormal):
    abnormal_train = abnormal[abnormal_train_index]
    abnormal_test = abnormal[abnormal_test_index]


    if not os.path.exists(folder+"train_kfold"+str(i)) and not os.path.exists(folder+"test_kfold"+str(i)):
        os.mkdir(folder+"train_kfold"+str(i))
        os.mkdir(folder+"test_kfold"+str(i))

    if (not os.path.exists(folder+"train_kfold"+str(i)+"/abnormal") and not os.path.exists(folder+"test_kfold"+str(i)+"/abnormal")):
        os.mkdir(folder+"train_kfold"+str(i)+"/abnormal")
        os.mkdir(folder+"test_kfold"+str(i)+"/abnormal")

    copyTrain(abnormal_train, i)
    copyTest(abnormal_test, i)
    
    i+=1
    


In [ ]:
import librosa.display
import scipy.io.wavfile as wavfile
import numpy
import os.path
from os import walk
from scipy import stats
import numpy as np
import librosa 
import numpy as np
from scipy.stats import norm
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
# Import the libraries
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import svm
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import confusion_matrix
from pycm import *
import numpy as np

def extractMelSpectrogram_features(folder, selection_nb):
    hop_length = 512
    n_fft = 2048
    n_mels = 128
    types = ["abnormal", "normal"]
    a = []
    b = []
    for nametype in types:
        _wavs = []
        wavs_duration = []
        for (_,_,filenames) in walk(folder+nametype+"/"):
            _wavs.extend(filenames)
            break
        Mel_Spectrogram = []
        _wavs = _wavs[:selection_nb]
        print (_wavs)
        for _wav in _wavs:
            # read audio samples
            if(".wav" in _wav): 
                file = folder +nametype+"/"+_wav
                print ("-"+file)
                signal, rate = librosa.load(file)  
                #The Mel Spectrogram
                S = librosa.feature.melspectrogram(signal, sr=rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
                S_DB = librosa.power_to_db(S, ref=np.max)
                #Mel_Spectrogram.append(S_DB)
                #print (S_DB)
                S_DB = S_DB.flatten()[:1200]
                a.append(S_DB)
                if(nametype == "normal"):
                    b.append(0)
                elif (nametype == "abnormal"):
                    b.append(1)
    pickle.dump(a, open(folder+'mel_features_svm_a.data', 'wb'))
    pickle.dump(b, open(folder+'mel_features_svm_b.data', 'wb'))
    
    return a, b

def getPrediction(folder, clf, selection_nb):
    hop_length = 512
    n_fft = 2048
    n_mels = 128
    types = ["abnormal", "normal"]
    list_y_true = []
    list_y_pred = []
    for nametype in types:
        _wavs = []
        wavs_duration = []
        for (_,_,filenames) in walk(folder+nametype+"/"):
            _wavs.extend(filenames)
            break
        _wavs = _wavs[:selection_nb]
        for _wav in _wavs:
            # read audio samples
            if(".wav" in _wav): 
                file = folder +nametype+"/"+_wav
                print ("-"+file)
                signal, rate = librosa.load(file)  
                #The Mel Spectrogram
                S = librosa.feature.melspectrogram(signal, sr=rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
                S_DB = librosa.power_to_db(S, ref=np.max)
                S_DB = S_DB.flatten()[:1200]
                y_pred = clf.predict([S_DB])[0]
                if(nametype=="abnormal"):
                    y_true = 1
                else:
                    y_true = 0
                list_y_true.append(y_true)
                list_y_pred.append(y_pred)
                print ("- class: "+str(nametype)+" = "+str(y_true)+", prediction: "+str(y_pred))
    pickle.dump(list_y_true, open(folder+'list_y_true.data', 'wb'))
    pickle.dump(list_y_pred, open(folder+'list_y_pred_svm_prediction.data', 'wb'))
    return list_y_true, list_y_pred
    #
#---
folder = "/content/drive/MyDrive/MachineLearning/ourdataset/train/"
types = ["abnormal", "normal"]

kfolds = np.arange(3)+1
list_macro_precision = []
list_mean_macro_precision = []
selection_nb = 5
for selection_nb in [5, 10, 15, 20]:
  for k in kfolds:
    print ("########## - Process KFold"+str(k))
    a, b = extractMelSpectrogram_features(folder+"train_kfold"+str(k)+"/", selection_nb)

    print ("#Load mel features:")
    a = pickle.load(open(folder+'mel_features_svm_a.data', 'rb'))
    b = pickle.load(open(folder+'mel_features_svm_b.data', 'rb'))
    print ("#SVM:")
    clf = svm.SVC()
    clf = svm.SVC(kernel="rbf")
    clf.fit(a, b)
    list_y_true, list_y_pred = getPrediction(folder+"test_kfold"+str(k)+"/", clf, selection_nb)


    list_y_true = pickle.load(open(folder+"test_kfold"+str(k)+"/"+'list_y_true.data', 'rb'))
    list_y_pred = pickle.load(open(folder+"test_kfold"+str(k)+"/"+'list_y_pred_svm_prediction.data', 'rb'))

    cl_report = classification_report(list_y_true, list_y_pred, target_names=types, output_dict=True)
    macro_precision =  cl_report['macro avg']['precision']
    list_macro_precision.append(macro_precision)
    print ("- Macro avg Precision:"+str(macro_precision))

    results = folder+"kfold"+str(k)+"_report.txt"
    f = open(results, "w")
    cm = ConfusionMatrix(actual_vector=list_y_true, predict_vector=list_y_pred) # Create CM From Data
    f.write("{}\n".format(cm))
    f.close()
  mean_macro_precision = np.mean(list_macro_precision)
  list_mean_macro_precision.append(mean_macro_precision)
  print ("##############---- wavs selection number from dataset: "+str(selection_nb))
  print ("#######---- Mean macro precision: "+str(mean_macro_precision))

import matplotlib.pyplot as plt


plt.plot(np.arange(len(list_mean_macro_precision))+1, list_mean_macro_precision)
plt.xlabel('wavs selection number from dataset')
plt.ylabel('Mean macro precision')
plt.show()



########## - Process KFold1
['d0012.wav', '116_1306258689913_A.wav', 'a0157.wav', 'a0180.wav', '201101051108.wav']
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/abnormal/d0012.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/abnormal/116_1306258689913_A.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/abnormal/a0157.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/abnormal/a0180.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/abnormal/201101051108.wav
['e01039.wav', 'e01058.wav', 'e00804.wav', 'e01063.wav', 'e01056.wav']
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/normal/e01039.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/normal/e01058.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/normal/e00804.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold1/normal/e01063.

/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/abnormal/b0043.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/abnormal/a0180.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/abnormal/201101051108.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/abnormal/a0197.wav
['e01102.wav', 'e00954.wav', 'e00807.wav', 'e01108.wav', 'e00883.wav']
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/normal/e01102.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/normal/e00954.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/normal/e00807.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/normal/e01108.wav
-/content/drive/MyDrive/MachineLearning/ourdataset/train/train_kfold2/normal/e00883.wav
#Load mel features:
#SVM:
-/content/drive/MyDrive/MachineLearning/ourdataset/train/test_kfold2/abnormal/116_1306258689913_A.wa